In [1]:
import tensorflow as tf
import os
import json
import subprocess
import glob
from difflib import SequenceMatcher

2025-04-18 06:03:51.577428: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-18 06:03:51.588946: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744952631.598216     474 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744952631.600849     474 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-18 06:03:51.616603: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
def parse_song_name(example):
    feature_description = {
        'labels': tf.io.VarLenFeature(tf.int64),
        'mel_spectrogram': tf.io.FixedLenFeature([], tf.string),
        'song_name': tf.io.FixedLenFeature([], tf.string),
        'segment_idx': tf.io.FixedLenFeature([], tf.int64),
        'total_segments': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example, feature_description)
    return parsed['song_name']

In [3]:
def string_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

In [4]:
def get_test_songs(segment_length=3): #all the segment version datasets are the same
    test_dir = os.path.join(f"../creating_spectrogram_batches/tfrecord_dataset_{segment_length}s", "test")
    if not os.path.exists(test_dir):
        print(f"Warning: Test directory not found at {test_dir}")
        return set()
    
    tfrecord_files = glob.glob(os.path.join(test_dir, "*.tfrecord"))
    if not tfrecord_files:
        print(f"Warning: No TFRecord files found in {test_dir}")
        return set()
    
    raw_dataset = tf.data.TFRecordDataset(tfrecord_files)
    test_songs = set()
    
    for raw_record in raw_dataset:
        song_name = parse_song_name(raw_record).numpy().decode('utf-8')
        test_songs.add(song_name)
    
    print(f"Loaded {len(test_songs)} test songs to avoid overlap")
    return test_songs

In [5]:
def might_overlap_with_test_songs(query, test_songs, similarity_threshold=0.8):
    query_lower = query.lower()
    
    # First check for direct substring matches (faster)
    for test_song in test_songs:
        test_song_lower = test_song.lower()
        
        # Check if the test song title is contained in the query
        if test_song_lower in query_lower:
            print(f"Overlap detected - Test song '{test_song}' found in query '{query}'")
            return True
            
    # For non-obvious matches, use similarity ratio
    for test_song in test_songs:
        similarity = string_similarity(query.split(' ')[0], test_song)  # Compare just the title part
        if similarity > similarity_threshold:
            print(f"Similar song detected - Test: '{test_song}', Query: '{query}', Similarity: {similarity:.2f}")
            return True
    
    return False

In [6]:
test_songs = get_test_songs()

I0000 00:00:1744952934.678849     474 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5520 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2025-04-18 06:08:54.784542: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
2025-04-18 06:09:01.897787: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Loaded 160 test songs to avoid overlap


In [7]:
with open('../json/track_ids.json', 'r') as f:
    track_id_title_artists = json.load(f)

In [8]:
track_title_artists = [(title, artist) for id_, title, artist in track_id_title_artists] #dropping the id

In [9]:
track_title_artists_concat = [title + ' ' + ''.join(str(artist) + ' ' for artist in artists) for title, artists in track_title_artists]

In [10]:
track_title_artists_concat

['Pentafilla Kai Paparounes Dof Twogee Sadam ',
 'Pričaj Mi O Ljubavi Neno Belan Djavoli ',
 'The Fog and the Grog Alfred Drake Christopher Hewett Robert Penn Arthur Rubin Ensemble ',
 'Fabulous - Mixed Anna Vissi Nikos Halkousis ',
 'Verdi: Rigoletto, Act 1: "Gualtier Maldè… Caro nome" (Gilda, Borsa, Ceprano, Marullo) Giuseppe Verdi Maria Callas Carlo Forti Coro Del Teatro Alla Scala Di Milano Renato Ercolani William Dickie Tullio Serafin Orchestra Del Teatro Alla Scala, Milano ',
 'Baby Anemona Brainwave ',
 'The Marketplace (From "The Scarlet Letter") John Morris ',
 'Aggression Incarnate Phosgore ',
 'Pera Sto Tholo Potami Manos Hadjidakis Flery Dandonaki ',
 'Ρώσικη Ρουλέτα Marseaux WNCfam ',
 'Second Chance New Zero God & Friends ',
 'Stous Pedikous Mou Kipous Stavros Siolas Maria Papageorgiou ',
 'Χειμώνας Lexicon Project ',
 'An Den Orizis Esena Magic De Spell Iakovos Paterakis ',
 'To Soma Ayto Einai Deilo Diafana Krina ',
 'Pote Tha Ftasoume Edo Giannis Aggelakas Nikos Veliot

In [11]:
os.makedirs("greek_mp3", exist_ok=True)

# Download songs, skipping those with potential overlap
skipped_overlap = 0
skipped_exists = 0
downloaded = 0
failed = 0

In [14]:
for query in track_title_artists_concat:
    base_title = query.replace(" ", "_").replace("/", "_")
    filename = f"greek_mp3/{base_title}.mp3"
        
    # Skip if already downloaded
    if os.path.exists(filename):
        print(f"Skip (already exists): {filename}")
        skipped_exists += 1
        continue
        
        # Check for overlap with test dataset
    if might_overlap_with_test_songs(query, test_songs):
        print(f"Skip (potential test set overlap): {query}")
        skipped_overlap += 1
        continue
        
    # Build yt-dlp command
    cmd = [
        "yt-dlp",
        f"ytsearch1:{query}",
        "-x", "--audio-format", "mp3",
        "--output", filename,
        "--quiet", "--no-warnings"
    ]
        
    try:
        subprocess.run(cmd, check=True)
        downloaded += 1
    except subprocess.CalledProcessError:
        failed += 1

ERROR: Unable to download video: [Errno 36] File name too long: 'greek_mp3/Verdi:_Rigoletto,_Act_1:_"Gualtier_Maldè…_Caro_nome"_(Gilda,_Borsa,_Ceprano,_Marullo)_Giuseppe_Verdi_Maria_Callas_Carlo_Forti_Coro_Del_Teatro_Alla_Scala_Di_Milano_Renato_Ercolani_William_Dickie_Tullio_Serafin_Orchestra_Del_Teatro_Alla_Scala,_Milano_.mp4.ytdl'
ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe
ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe
ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe
ERROR: Unable to download video: [Errno 36] File name too long: "greek_mp3/Rolling_110_Deep_(feat._Sheek_Louch,_Styles_P,_Dave_East,_Crooked_I,_Black_Thought,_Raekwon,_Ghostface_Killah,_Inspectah_Deck,_Papoose,_Loaded_Lux,_AZ,_Bun_B,_Fred_the_Godson,_Jim_Jones,_Ransom,_Rah_Digga,_Billy_Danze,_Lil_Fame,_Trae_tha_Truth,_Joell_Ortiz,_Lord_Tariq,_Cory_Gunz,_Peter_Gunz,_Shaq_Diesel,_Roy_Jones_Jr.,_DJ_Red_Alert,_Redman,_Young_Buck

Overlap detected - Test song 'Anastasia' found in query 'To Tragoudi Tou Gamou Despina Stylianopoulou Alekos Anastasiadis Yiorgos Katsaros '
Skip (potential test set overlap): To Tragoudi Tou Gamou Despina Stylianopoulou Alekos Anastasiadis Yiorgos Katsaros 
Similar song detected - Test: 'Lola', Query: 'Ola Dika Mas Minos Matsas Rena Morfi Giannis Stankoglou ', Similarity: 0.86
Skip (potential test set overlap): Ola Dika Mas Minos Matsas Rena Morfi Giannis Stankoglou 
Similar song detected - Test: 'Lola', Query: 'Ola Se Thimizoun Haris Alexiou ', Similarity: 0.86
Skip (potential test set overlap): Ola Se Thimizoun Haris Alexiou 
Overlap detected - Test song 'Anastasia' found in query 'Ah Eleni Dimos Anastasiadis Zoi Papadopoulou '
Skip (potential test set overlap): Ah Eleni Dimos Anastasiadis Zoi Papadopoulou 
Overlap detected - Test song 'Stella' found in query 'Kardia paraponiara - Καρδιά παραπονιάρα Stella Haskil Takis Mpinis '
Skip (potential test set overlap): Kardia paraponiara -

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


Overlap detected - Test song 'Stella' found in query 'Hristougenna 2010 Elli Kokkinou Thanos Petrelis Nino Xypolitas Panos Kallidis Elisavet Spanou Ioakim Fokas Stella Kalli Paidiki Horodia Spyrou Labrou Foivos '
Skip (potential test set overlap): Hristougenna 2010 Elli Kokkinou Thanos Petrelis Nino Xypolitas Panos Kallidis Elisavet Spanou Ioakim Fokas Stella Kalli Paidiki Horodia Spyrou Labrou Foivos 


ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


...

Similar song detected - Test: 'Vradiazei', Query: 'Vradiazi Sto Katastroma - Orchestral Marios Strofalis ', Similarity: 0.94
Skip (potential test set overlap): Vradiazi Sto Katastroma - Orchestral Marios Strofalis 


ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


Skip (already exists): greek_mp3/Hello_Sunshine_-_Feat._Antonia_Jenae_The_Square_Egg_.mp3
Similar song detected - Test: 'Erwtiko', Query: 'Erotiko Hainides ', Similarity: 0.86
Skip (potential test set overlap): Erotiko Hainides 



ERROR: Interrupted by user


KeyboardInterrupt: 

In [15]:
# Print summary
print("\n==== Download Summary ====")
print(f"Total tracks processed: {len(track_title_artists_concat)}")
print(f"Already existed: {skipped_exists}")
print(f"Skipped due to test overlap: {skipped_overlap}")
print(f"Successfully downloaded: {downloaded}")
print(f"Failed downloads: {failed}")


==== Download Summary ====
Total tracks processed: 30587
Already existed: 92
Skipped due to test overlap: 89
Successfully downloaded: 5659
Failed downloads: 88
